# Classic models (CNN) — vodič kroz notebook

Isti zadatak — prepoznavanje kompozitora po djelu — ali ovdje se ne pravi ručni feature vektor. Umjesto toga,
svaki audio segment se pretvara u **log-mel spektrogram** (2D "slika": mel frekvencija × vrijeme), i mala CNN
mreža sama uči šta u toj slici razlikuje kompozitore.

Segmenti su ovdje **10s** (`config.SEGMENT_DURATION`), kraći nego kod klasičnih modela (25s) — više trening
primjera za istu količinu audija, bitno kad imamo samo 113 djela.

PyTorch je CPU-only na ovoj mašini (nema CUDA) — trening je zato sporiji nego na GPU-u; model je namjerno
mali da to ostane izvodljivo.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "..")

# Ucitavanje podataka

## Priprema metapodataka

Pregled izvorne tabele i raspodjele kompozitora nalazi se u [metadata.ipynb](metadata.ipynb).
`load_metadata()` iz `dataset.py` učitava CSV, formira `work_id` kao `composer + " | " + composition`
i zadržava kompozitore sa najmanje `MIN_WORKS_PER_COMPOSER` različitih djela (podrazumijevano 5, iz `config.py`).
Svi stavovi istog djela tako ostaju u istoj grupi tokom evaluacije.

Ova sveska se izvršava samostalno; nije potrebno prethodno pokretati `metadata.ipynb`.

In [ ]:
import numpy as np
import pandas as pd

from src.features.dataset import load_metadata
from src.utils.paths import AUDIO_DIR

metadata_filtered = load_metadata()

In [8]:
print(
    metadata_filtered.groupby("composer")["work_id"]
    .nunique()
    .sort_values()
)

composer
Brahms        8
Schubert      9
Mozart       11
Bach         30
Beethoven    55
Name: work_id, dtype: int64


---

## Mel-spektrogram dataset

`make_spectrogram_dataset` (u `spectrograms.py`) radi isto što i `make_dataset`/`make_midi_dataset` — učita
audio, isiječe na segmente, ali umjesto ručnih atributa računa `extract_melspec` (128 mel binova ×
~862 vremenskih frame-ova po segmentu, log-dB skala). Rezultat je keširan na disk (invalidira se automatski
ako se promijeni `spectrograms.py`).

Pošto su `SEGMENT_DURATION == MIN_LAST_DURATION == 10`, zadržavaju se samo puni 10s segmenti — svi imaju
identičnu dužinu, pa je `X` običan 3D numpy niz (N, 128, T), bez potrebe za paddingom.

In [ ]:
from src.features.spectrograms import make_spectrogram_dataset

X, y, groups = make_spectrogram_dataset(metadata_filtered, AUDIO_DIR)

print(X.shape)

---

## Train/val split

Pun 5-fold `StratifiedGroupKFold` (kao kod klasičnih modela) bi značio 5x treniranje CNN-a — na CPU-u
predugo za iterativno eksperimentisanje. Za sada radimo **jedan** group-aware split (`GroupShuffleSplit` —
segmenti istog `work_id` ne smiju biti i u train i u val). Kad se arhitektura/hiperparametri skoro finalizuju,
vrijedi ponoviti sa punim CV radi pouzdanije ocjene.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder

from src.model.evaluation import make_sample_weights, soft_vote_predictions

le = LabelEncoder()
y_enc = le.fit_transform(y)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X, y_enc, groups))

X_train, y_train, groups_train = X[train_idx], y_enc[train_idx], groups[train_idx]
X_val, y_val, groups_val = X[val_idx], y[val_idx], groups[val_idx]

sample_weights = make_sample_weights(y[train_idx], groups_train)

print("train segments:", len(X_train), "val segments:", len(X_val))

---

## Model i trening

Mali CNN (`SimpleCNN` u `nn_utils.py`): 3 conv+BN+ReLU+maxpool bloka, pa global average pooling i dropout
prije finalnog linearnog sloja — namjerno malo parametara zbog svega 113 djela. `SpecAugment` (nasumično
maskiranje frekvencijskih/vremenskih traka) je uključen samo na trening skupu, kao augmentacija.
`sample_weights` (isti mehanizam kao kod klasičnih modela) ide kroz `WeightedRandomSampler`, umjesto direktno
u loss funkciju.

In [ ]:
import torch

from src.model.nn_model import SpectrogramDataset, SimpleCNN, train_model, plot_history

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

train_ds = SpectrogramDataset(X_train, y_train, augment=True)
val_ds = SpectrogramDataset(X_val, le.transform(y_val), augment=False)

model = SimpleCNN(n_classes=len(le.classes_))

history = train_model(
    model, train_ds, val_ds,
    sample_weights=sample_weights,
    epochs=20,
    batch_size=32,
    lr=1e-3,
    device=device
)

plot_history(history)

---

## Work-level evaluacija

Model predviđa po segmentu; `soft_vote_predictions` (ista funkcija iz `evaluation.py` koju smo definisali za
klasične modele, do sad neiskorištena) prosječi softmax vjerovatnoće svih segmenata jednog djela i uzme
najvjerovatniju klasu — konzistentno sa `majority_vote_prediction` pristupom kod ostalih modela, samo
iskorišćava i "koliko je model siguran", ne samo tvrdu predikciju.

In [ ]:
from sklearn import metrics
from matplotlib import pyplot as plt

from src.model.nn_model import predict_probs

probs = predict_probs(model, X_val, device=device)

work_true, work_pred = soft_vote_predictions(y_val, probs, groups_val, le.classes_)

print(metrics.classification_report(work_true, work_pred))

metrics.ConfusionMatrixDisplay.from_predictions(work_true, work_pred, normalize="true")
plt.show()